# 02 - Skill Matcher (hybrid)
**Task:** the user types skills in free text ("py", "pyhton", "making charts from data").
We must map each one to a standard skill from `skills.csv`, or say "no match".

Three layers, tried in order (code in `skill_matcher.py`):
1. **Alias lookup**: normalized text matches a skill name or alias exactly
2. **Fuzzy match**: typos ("pyhton" -> Python), edit-distance based
3. **Embedding similarity**: meaning-based ("cleaning messy datasets" -> Data Cleaning)

Embedding matches with medium similarity are returned with `needs_confirmation = True`
so the website asks the user, instead of silently guessing.

In [1]:
import pandas as pd
from skill_matcher import SkillMatcher, normalize, split_skills

skills = pd.read_csv("../data/skills.csv")
test = pd.read_csv("../data/skill_match_test.csv").fillna("")
print(len(skills), "skills |", len(test), "test inputs")
test["kind"].value_counts()

42 skills | 90 test inputs


kind
alias     40
phrase    24
none      14
typo      12
Name: count, dtype: int64

## Test set
`skill_match_test.csv` has 4 kinds of input:
- **alias**: known short forms ("js", "sklearn")
- **typo**: spelling mistakes ("javscript")
- **phrase**: descriptions, not skill names ("writing unit tests")
- **none**: skills we do NOT have ("photoshop", "java"); the correct answer is *no match*

In [2]:
def outcome(row):
    exp = row["expected"]
    got = "" if pd.isna(row["skill_id"]) else row["skill_id"]
    if exp == "":
        if row["method"] == "none":
            return "correct_none"
        return "false_ask" if row["needs_confirmation"] else "false_match"
    if got == exp:
        return "correct_asked" if row["needs_confirmation"] else "correct_auto"
    if got == "":
        return "missed"
    return "wrong_asked" if row["needs_confirmation"] else "wrong_auto"

def evaluate(matcher):
    res = pd.DataFrame(matcher.match_many(test["raw_text"]))
    res["expected"] = test["expected_skill_id"].values
    res["kind"] = test["kind"].values
    res["outcome"] = res.apply(outcome, axis=1)
    return res

def summary(res):
    c = res["outcome"].value_counts()
    cols = ["correct_auto", "correct_asked", "correct_none", "missed", "wrong_asked", "wrong_auto", "false_ask", "false_match"]
    out = {k: int(c.get(k, 0)) for k in cols}
    out["good_%"] = round(100 * (out["correct_auto"] + out["correct_asked"] + out["correct_none"]) / len(res), 1)
    out["bad_auto"] = out["wrong_auto"] + out["false_match"]   # silent mistakes: the worst kind
    return out

## 1. Ablation: what does each layer add?
- **correct_auto**: right answer, accepted automatically
- **correct_asked**: right answer, but the user is asked to confirm
- **correct_none**: correctly said "no match"
- **missed**: should have matched but returned nothing
- **bad_auto**: silent mistakes (wrong skill accepted, or a fake match) - this must stay near 0

In [3]:
A = evaluate(SkillMatcher(skills, fuzzy_threshold=None))
B = evaluate(SkillMatcher(skills))
pd.DataFrame({"A: alias only": summary(A), "B: alias + fuzzy": summary(B)}).T

,correct_auto,correct_asked,correct_none,missed,wrong_asked,wrong_auto,false_ask,false_match,good_%,bad_auto
A: alias only,39.0,0.0,14.0,37.0,0.0,0.0,0.0,0.0,58.9,0.0
B: alias + fuzzy,51.0,0.0,14.0,25.0,0.0,0.0,0.0,0.0,72.2,0.0


In [4]:
# Which kinds of input does each setup handle?
pd.DataFrame({
    "A": A.groupby("kind")["outcome"].apply(lambda s: s.isin(["correct_auto", "correct_none"]).mean()),
    "B": B.groupby("kind")["outcome"].apply(lambda s: s.isin(["correct_auto", "correct_none"]).mean()),
}).round(2)

,A,B
kind,,
alias,0.98,1.00
none,1.00,1.00
phrase,0.00,0.00
typo,0.00,0.92


## 2. Fuzzy threshold
Lower = catches more typos but risks false matches. We want the lowest value with `bad_auto = 0`.

In [5]:
rows = []
for th in [95, 90, 85, 80, 75, 70]:
    s = summary(evaluate(SkillMatcher(skills, fuzzy_threshold=th)))
    rows.append({"fuzzy_threshold": th, **s})
pd.DataFrame(rows)

,fuzzy_threshold,correct_auto,correct_asked,correct_none,missed,wrong_asked,wrong_auto,false_ask,false_match,good_%,bad_auto
0,95,39,0,14,37,0,0,0,0,58.9,0
1,90,49,0,14,27,0,0,0,0,70.0,0
2,85,49,0,14,27,0,0,0,0,70.0,0
3,80,51,0,14,25,0,0,0,0,72.2,0
4,75,55,0,14,21,0,0,0,0,76.7,0
5,70,57,0,13,19,0,0,0,1,77.8,1


## 3. Embedding layer
Needs `pip install sentence-transformers`. The first run downloads a small model (~90 MB).
If it fails, the notebook keeps working with layers 1-2 only.

In [6]:
try:
    from skill_matcher import load_sentence_encoder
    encoder = load_sentence_encoder("all-MiniLM-L6-v2")
    print("embedding model loaded")
except Exception as e:
    encoder = None
    print("Embedding model not available, skipping layer 3:", type(e).__name__, e)

embedding model loaded


In [7]:
if encoder is not None:
    mC = SkillMatcher(skills, encoder=encoder)
    C = evaluate(mC)
    display(pd.DataFrame({"A: alias only": summary(A), "B: alias + fuzzy": summary(B),
                          "C: alias + fuzzy + embedding": summary(C)}).T)

,correct_auto,correct_asked,correct_none,missed,wrong_asked,wrong_auto,false_ask,false_match,good_%,bad_auto
A: alias only,39.0,0.0,14.0,37.0,0.0,0.0,0.0,0.0,58.9,0.0
B: alias + fuzzy,51.0,0.0,14.0,25.0,0.0,0.0,0.0,0.0,72.2,0.0
C: alias + fuzzy + embedding,71.0,2.0,8.0,1.0,2.0,0.0,5.0,1.0,90.0,1.0


## 4. Embedding thresholds
- similarity >= `emb_high`: accepted automatically
- `emb_low` <= similarity < `emb_high`: the user is asked to confirm
- below `emb_low`: no match

Pick the setting where `bad_auto` is lowest while `good_%` stays high.

In [8]:
if encoder is not None:
    rows = []
    for high in [0.5, 0.6, 0.7, 0.8]:
        for low in [0.3, 0.4, 0.5]:
            if low >= high:
                continue
            mC.emb_high, mC.emb_low = high, low
            rows.append({"emb_high": high, "emb_low": low, **summary(evaluate(mC))})
    display(pd.DataFrame(rows))

,emb_high,emb_low,correct_auto,correct_asked,correct_none,missed,wrong_asked,wrong_auto,false_ask,false_match,good_%,bad_auto
0,0.5,0.3,72,1,1,0,1,2,11,2,82.2,4
1,0.5,0.4,72,1,8,1,0,2,4,2,90.0,4
2,0.6,0.3,71,2,1,0,3,0,12,1,82.2,1
3,0.6,0.4,71,2,8,1,2,0,5,1,90.0,1
4,0.6,0.5,71,1,12,2,2,0,1,1,93.3,1
5,0.7,0.3,67,6,1,0,3,0,13,0,82.2,0
6,0.7,0.4,67,6,8,1,2,0,6,0,90.0,0
7,0.7,0.5,67,5,12,2,2,0,2,0,93.3,0
8,0.8,0.3,58,15,1,0,3,0,13,0,82.2,0
9,0.8,0.4,58,15,8,1,2,0,6,0,90.0,0


## 5. Where does it still fail?

In [9]:
final = C if encoder is not None else B
final[final["outcome"].isin(["missed", "wrong_auto", "wrong_asked", "false_match", "false_ask"])][
    ["raw_text", "expected", "skill_id", "method", "score", "outcome"]]

,raw_text,expected,skill_id,method,score,outcome
44,htlm,S04,S37,embedding,0.543,wrong_asked
53,making websites look good,S05,S04,embedding,0.549,wrong_asked
73,accuracy precision and recall,S26,NaN,none,0.375,missed
76,excel,,S21,embedding,0.473,false_ask
77,photoshop,,S05,embedding,0.468,false_ask
81,java,,S06,embedding,0.606,false_match
82,c++,,S29,embedding,0.507,false_ask
83,ms word,,S04,embedding,0.442,false_ask
88,kotlin,,S24,embedding,0.404,false_ask


## 6. Try your own inputs
Users often type several skills in one box, so the backend first splits the text with `split_skills`.

In [10]:
print(split_skills("python, html and css & sql"))

['python', 'html', 'css', 'sql']


In [11]:
matcher = mC if encoder is not None else SkillMatcher(skills)
for t in [x for field in ["py, Pyhton", "html and css", "machine learnin", "making apps talk to servers", "photoshop, excel"] for x in split_skills(field)]:
    r = matcher.match(t)
    print(f"{t!r:35} -> {r['skill_name']} | {r['method']} | score={r['score']} | confirm={r['needs_confirmation']}")

'py'                                -> Python | alias | score=1.0 | confirm=False
'Pyhton'                            -> Python | fuzzy | score=0.833 | confirm=False
'html'                              -> HTML | alias | score=1.0 | confirm=False
'css'                               -> CSS | alias | score=1.0 | confirm=False
'machine learnin'                   -> Machine Learning Basics | fuzzy | score=0.968 | confirm=False
'making apps talk to servers'       -> None | none | score=0.315 | confirm=False
'photoshop'                         -> None | none | score=0.468 | confirm=False
'excel'                             -> None | none | score=0.473 | confirm=False
